# Lobbying Influence Analysis — 2025/0102(COD)

Assess whether pre-proposal engagement by the most active lobbying organisations
is textually reflected in the Commission proposal.

**Pipeline overview**

| Stage | What gets embedded | Source |
|---|---|---|
| 0 | Proposal articles (recitals + articles) | `procedure_articles` or `procedure_documents` |
| 0.1 | Amendments | `procedure_amendments` |
| 1 | Pre-proposal commission meeting points | `commission_meetings.points_raised` |
| 1.1 | Pre-proposal lobbying meeting purposes | `lobbying_meetings.meeting_purpose` |
| 1.2 | Pre-proposal HYS feedback chunks | `hys_feedback_chunks` |

**Stage 1 analysis**
1. Find the top `TOP_ORGS` organisations by pre-proposal meeting count.
2. For each org, run **top-3 reciprocal semantic matching** between their pre-proposal
   texts and the proposal articles.
3. Send matches to an LLM, which assesses whether the org's positions were
   reflected, opposed, or only coincidentally overlap with the proposal.

In [182]:
# ── Config — change these to rerun the analysis for a different procedure ──────
PROCEDURE_ID   = "2021/0106(COD)"

# Embedding model (all must be normalized dot-product compatible)
MODEL_NAME     = "sentence-transformers/all-mpnet-base-v2"
QUERY_PREFIX   = ""   # prepended to article text at query time
PASSAGE_PREFIX = ""   # prepended to meeting/chunk text

# Retrieval
TOP_K       = 10   # candidate pool before reciprocal filter
RECIP_TOP_N = 3    # max reciprocal matches sent to LLM per source type
TOP_ORGS    = 10    # how many top orgs to analyse

# LLM
LLM_MODEL = "claude-sonnet-4-6"

In [2]:
import os
import re
from collections import defaultdict

import numpy as np
import anthropic
from dotenv import load_dotenv
from supabase import create_client
from sentence_transformers import SentenceTransformer

load_dotenv()
supabase = create_client(os.getenv("SUPABASE_URL"), os.getenv("SUPABASE_SERVICE_ROLE_KEY"))
anth = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))
print("Connected to Supabase and Anthropic.")
print(f"  ANTHROPIC_API_KEY set: {bool(os.getenv('ANTHROPIC_API_KEY'))}")

/Users/inge/Desktop/ITU/Thesis/lobby_influence/EU_LOBBYING_INFLUENCE/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Connected to Supabase and Anthropic.
  ANTHROPIC_API_KEY set: True


In [3]:
# ── Helpers ───────────────────────────────────────────────────────────────────

def paginate(table, select, filter_fn=None, page_size=1000):
    """Fetch all rows from a Supabase table, bypassing the default 1000-row limit."""
    rows, offset = [], 0
    while True:
        q = supabase.table(table).select(select)
        if filter_fn:
            q = filter_fn(q)
        page = q.range(offset, offset + page_size - 1).execute().data or []
        rows.extend(page)
        if len(page) < page_size:
            break
        offset += page_size
    return rows


def batch_in(table, select, column, values, extra_filter_fn=None, batch_size=100):
    """Fetch rows where `column IN values`, batched to avoid URL-length limits."""
    rows = []
    for i in range(0, len(values), batch_size):
        batch = values[i : i + batch_size]
        q = supabase.table(table).select(select).in_(column, batch)
        if extra_filter_fn:
            q = extra_filter_fn(q)
        rows.extend(q.execute().data or [])
    return rows


print("Helpers ready.")

Helpers ready.


In [16]:
# ── Procedure metadata ────────────────────────────────────────────────────────
proc = (
    supabase.table("procedures")
    .select("title, proposal_date, commission_document")
    .eq("id", PROCEDURE_ID)
    .single()
    .execute()
    .data
)

TITLE         = proc["title"]
PROPOSAL_DATE = proc["proposal_date"]
# PROPOSAL_DATE = "2026-01-21"
COM_DOC       = proc.get("commission_document") or ""

print(f"Procedure:     {PROCEDURE_ID}")
print(f"Title:         {TITLE}")
print(f"Proposal date: {PROPOSAL_DATE}")
print(f"COM document:  {COM_DOC}")

Procedure:     2025/0102(COD)
Title:         Critical Medicines Act
Proposal date: 2025-03-11
COM document:  COM(2025)0102


## 0. Proposal Articles

In [8]:
# Try structured procedure_articles table first; fall back to raw document split
art_rows = paginate(
    "procedure_articles",
    "element_type, element_number, title, content, sort_order",
    lambda q: (
        q.eq("procedure_id", PROCEDURE_ID)
         .eq("document_version", "proposal")
         .order("sort_order")
    ),
)

if art_rows:
    proposal_articles = art_rows
    print(f"Loaded {len(proposal_articles)} elements from procedure_articles table.")
else:
    print("procedure_articles empty for this procedure — falling back to document split.")

    doc_row = (
        supabase.table("procedure_texts")
        .select("proposal_id, proposal_text")
        .eq("procedure_id", PROCEDURE_ID)
        .not_.is_("proposal_text", "null")
        .limit(1)
        .execute()
        .data or []
    )
    if not doc_row:
        raise ValueError(f"No commission_proposal text found for {PROCEDURE_ID}")

    doc_text = doc_row[0]["proposal_text"]
    print(f"Splitting document ({len(doc_text):,} chars)...")

    def split_into_articles(text):
        elements = []
        # Recitals block
        whereas = re.search(
            r'(?:Whereas|WHEREAS)[:\s]+(.*?)(?=HAS ADOPTED|HAVE ADOPTED|HEREBY ADOPTS)',
            text, re.DOTALL | re.IGNORECASE,
        )
        if whereas:
            block = whereas.group(1)
            hits  = list(re.finditer(r'(?:^|\n)\s*\((\d+)\)\s+', block))
            for i, m in enumerate(hits):
                end     = hits[i + 1].start() if i + 1 < len(hits) else len(block)
                content = re.sub(r'\s+', ' ', block[m.end():end]).strip()
                if len(content) > 40:
                    elements.append({
                        "element_type":   "recital",
                        "element_number": str(int(m.group(1))),
                        "title":          None,
                        "content":        content,
                    })
        # Articles
        art_pat = re.compile(
            r'(?:^|\n)[ \t]*Article[ \t]+(\d+)[ \t]*([^\n]{0,80})\n(.*?)(?=(?:^|\n)[ \t]*Article[ \t]+\d+|\Z)',
            re.DOTALL | re.MULTILINE,
        )
        for m in art_pat.finditer(text):
            num   = int(m.group(1))
            title = m.group(2).strip()
            body  = re.sub(r'\s+', ' ', m.group(3)).strip()
            if num > 200 or len(body) < 40:
                continue
            if re.search(r'\bTFEU\b|\bTEU\b|\bCharter\b', title, re.IGNORECASE):
                continue
            elements.append({
                "element_type":   "article",
                "element_number": str(num),
                "title":          title or None,
                "content":        body[:5000],
            })
        return elements

    proposal_articles = split_into_articles(doc_text)

recitals_n = sum(1 for a in proposal_articles if a["element_type"] == "recital")
arts_n     = sum(1 for a in proposal_articles if a["element_type"] == "article")
print(f"Result: {recitals_n} recitals, {arts_n} articles.")

procedure_articles empty for this procedure — falling back to document split.
Splitting document (137,216 chars)...
Result: 46 recitals, 50 articles.


In [9]:
# ── Load embedding model & embed proposal articles ────────────────────────────
print(f"Loading {MODEL_NAME}...")
model = SentenceTransformer(MODEL_NAME)
DIM   = model.get_sentence_embedding_dimension()
print(f"Embedding dimension: {DIM}")


def embed(texts, prefix=""):
    """Encode a list of strings; returns (n, DIM) float32 ndarray."""
    if not texts:
        return np.empty((0, DIM), dtype=np.float32)
    if prefix:
        texts = [prefix + t for t in texts]
    return model.encode(
        texts,
        batch_size=32,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True,
    )


def article_to_text(a):
    header = f"{a['element_type']} {a['element_number']}"
    if a.get("title"):
        header += f": {a['title']}"
    return f"{header}\n{a.get('content', '')}".strip()


article_texts = [article_to_text(a) for a in proposal_articles]
print(f"\nEmbedding {len(article_texts)} proposal elements...")
article_emb = embed(article_texts, prefix=QUERY_PREFIX)
print(f"Shape: {article_emb.shape}")

Loading sentence-transformers/all-mpnet-base-v2...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8470.68it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding dimension: 768

Embedding 96 proposal elements...


Batches: 100%|██████████| 3/3 [00:02<00:00,  1.04it/s]

Shape: (96, 768)


## 1. Pre-Proposal Commission Meetings

In [183]:
CM_COLS = "id, commissioner_name, commissioner_portfolio, meeting_date, subject, points_raised, conclusions"

print(f"Fetching commission meetings linked to {PROCEDURE_ID} before {PROPOSAL_DATE}...")

# JOIN meeting_procedure_links → commission_meetings at DB level, paginate all rows
meetings_by_id = {}
offset    = 0
page_size = 1000
while True:
    page = (
        supabase.table("meeting_procedure_links")
        .select(f"commission_meetings({CM_COLS})")
        .eq("procedure_id", PROCEDURE_ID)
        .not_.is_("commission_meeting_id", "null")
        .range(offset, offset + page_size - 1)
        .execute()
        .data or []
    )
    for row in page:
        m = row.get("commission_meetings")
        if m and m.get("id"):
            meetings_by_id[m["id"]] = m
    if len(page) < page_size:
        break
    offset += page_size

cm_rows_all = list(meetings_by_id.values())
print(f"  Link rows fetched:              {offset + len(page)}")
print(f"  Unique meetings (no date filter): {len(cm_rows_all)}")

cm_rows = [r for r in cm_rows_all if r.get("meeting_date") and r["meeting_date"] < PROPOSAL_DATE]
print(f"  Meetings before {PROPOSAL_DATE}:   {len(cm_rows)}")

cm_ids = [r["id"] for r in cm_rows]
cm_org_rows = batch_in(
    "commission_meeting_organizations",
    "meeting_id, organization_name, eu_transparency_register_id",
    "meeting_id", cm_ids,
) if cm_ids else []

cm_meeting_to_orgs = defaultdict(list)
for o in cm_org_rows:
    cm_meeting_to_orgs[o["meeting_id"]].append({
        "name":  o.get("organization_name") or "",
        "tr_id": o.get("eu_transparency_register_id"),
    })

print(f"  Organisation associations loaded: {len(cm_org_rows)}")

Fetching commission meetings linked to 2021/0106(COD) before 2021-04-21...
  Link rows fetched:              1169
  Unique meetings (no date filter): 960
  Meetings before 2021-04-21:   175
  Organisation associations loaded: 99


In [14]:
# Embed commission meeting content
def cm_to_text(m):
    parts = []
    if m.get("subject"):
        parts.append(f"Subject: {m['subject']}")
    if m.get("points_raised"):
        parts.append(m["points_raised"])
    if m.get("conclusions"):
        parts.append(f"Conclusions: {m['conclusions']}")
    return " ".join(parts).strip()


cm_valid_pairs = [(m, cm_to_text(m)) for m in cm_rows if len(cm_to_text(m)) > 20]
cm_rows_v  = [m for m, _ in cm_valid_pairs]
cm_texts_v = [t for _, t in cm_valid_pairs]

if cm_texts_v:
    print(f"Embedding {len(cm_texts_v)} commission meeting texts...")
    cm_emb = embed(cm_texts_v, prefix=PASSAGE_PREFIX)
    print(f"Shape: {cm_emb.shape}")
else:
    cm_emb = None
    print("No substantive commission meeting content found — cm_emb = None.")

Embedding 81 commission meeting texts...


Batches: 100%|██████████| 3/3 [00:03<00:00,  1.28s/it]

Shape: (81, 768)


## 1.1 Pre-Proposal Lobbying Meetings

In [ ]:
print(f"Fetching lobbying meetings linked to {PROCEDURE_ID} before {PROPOSAL_DATE}...")

LM_COLS = "id, organization_id, meeting_date, title, meeting_purpose, capacity"

# Same paginated JOIN approach as commission meetings
lm_by_id  = {}
offset    = 0
page_size = 1000
while True:
    page = (
        supabase.table("meeting_procedure_links")
        .select(f"lobbying_meetings({LM_COLS})")
        .eq("procedure_id", PROCEDURE_ID)
        .not_.is_("lobbying_meeting_id", "null")
        .range(offset, offset + page_size - 1)
        .execute()
        .data or []
    )
    for row in page:
        m = row.get("lobbying_meetings")
        if m and m.get("id"):
            lm_by_id[m["id"]] = m
    if len(page) < page_size:
        break
    offset += page_size

lm_rows_all = list(lm_by_id.values())
print(f"  Unique lobbying meetings (no date filter): {len(lm_rows_all)}")

lm_rows = [r for r in lm_rows_all if r.get("meeting_date") and r["meeting_date"] < PROPOSAL_DATE]
print(f"  Lobbying meetings (pre-proposal): {len(lm_rows)}")

# Resolve org names
lm_org_ids = list({r["organization_id"] for r in lm_rows if r.get("organization_id")})
lm_orgs_map = {}
if lm_org_ids:
    orgs = batch_in("organizations", "id, name, eu_transparency_register_id", "id", lm_org_ids)
    lm_orgs_map = {o["id"]: o for o in orgs}
    print(f"  Org details resolved: {len(lm_orgs_map)}")

In [ ]:
# Embed lobbying meeting content
def lm_to_text(m):
    parts = []
    if m.get("title"):
        parts.append(f"Meeting: {m['title']}")
    if m.get("meeting_purpose"):
        parts.append(m["meeting_purpose"])
    return " ".join(parts).strip()


lm_valid_pairs = [(m, lm_to_text(m)) for m in lm_rows if len(lm_to_text(m)) > 20]
lm_rows_v  = [m for m, _ in lm_valid_pairs]
lm_texts_v = [t for _, t in lm_valid_pairs]

if lm_texts_v:
    print(f"Embedding {len(lm_texts_v)} lobbying meeting texts...")
    lm_emb = embed(lm_texts_v, prefix=PASSAGE_PREFIX)
    print(f"Shape: {lm_emb.shape}")
else:
    lm_emb = None
    print("No substantive lobbying meeting content — lm_emb = None.")

## 1.2 Pre-Proposal HYS Feedback

In [ ]:
print(f"Fetching feedback chunks for {PROCEDURE_ID} before {PROPOSAL_DATE}...")

fb_rows = paginate(
    "hys_feedback_chunks",
    "id, chunk_text, organisation_name, transparency_reg_id, date_feedback",
    lambda q: q.eq("procedure_id", PROCEDURE_ID).lt("date_feedback", PROPOSAL_DATE),
)
print(f"Pre-proposal feedback chunks (all): {len(fb_rows)}")

# ── Build sets of orgs/TR IDs that have at least one meeting ──────────────────
meeting_org_names = {
    o["name"].lower()
    for orgs in cm_meeting_to_orgs.values()
    for o in orgs
    if o.get("name")
}
meeting_tr_ids = {
    o["tr_id"]
    for orgs in cm_meeting_to_orgs.values()
    for o in orgs
    if o.get("tr_id")
}
meeting_org_names |= {
    org.get("name", "").lower()
    for org in lm_orgs_map.values()
    if org.get("name")
}
meeting_tr_ids |= {
    org["eu_transparency_register_id"]
    for org in lm_orgs_map.values()
    if org.get("eu_transparency_register_id")
}

# ── Filter: keep only chunks from orgs with meetings ─────────────────────────
def has_meeting(row):
    tr = row.get("transparency_reg_id")
    name = (row.get("organisation_name") or "").lower()
    return (tr and tr in meeting_tr_ids) or (name and name in meeting_org_names)

fb_rows = [r for r in fb_rows if has_meeting(r)]
print(f"After filtering to orgs with meetings: {len(fb_rows)}")

# Show org distribution
fb_org_counts = defaultdict(int)
for r in fb_rows:
    fb_org_counts[r.get("organisation_name") or "(unnamed)"] += 1
print(f"\nUnique orgs in feedback: {len(fb_org_counts)}")
for name, count in sorted(fb_org_counts.items(), key=lambda x: x[1], reverse=True)[:10]:
    print(f"  {count:>5}  {name}")

if fb_rows:
    fb_texts = [r["chunk_text"] for r in fb_rows]
    print(f"\nEmbedding {len(fb_texts)} feedback chunks...")
    fb_emb = embed(fb_texts, prefix=PASSAGE_PREFIX)
    print(f"Shape: {fb_emb.shape}")
else:
    fb_texts, fb_emb = [], None
    print("No pre-proposal feedback found for orgs with meetings — fb_emb = None.")

## Stage 1 — Top Orgs by Pre-Proposal Meeting Count

In [137]:
org_meeting_counts = defaultdict(int)
org_tr_ids         = {}

# Commission meetings — count per org
for m in cm_rows:
    for o in cm_meeting_to_orgs.get(m["id"], []):
        name = o["name"]
        if name:
            org_meeting_counts[name] += 1
            if o.get("tr_id"):
                org_tr_ids[name] = o["tr_id"]

# Lobbying meetings — count per org
for m in lm_rows_v:
    org_id = m.get("organization_id")
    if org_id and org_id in lm_orgs_map:
        org  = lm_orgs_map[org_id]
        name = org.get("name") or ""
        if name:
            org_meeting_counts[name] += 1
            if org.get("eu_transparency_register_id"):
                org_tr_ids[name] = org["eu_transparency_register_id"]

sorted_orgs = sorted(org_meeting_counts.items(), key=lambda x: x[1], reverse=True)

print(f"{'Rank':<5} {'Meetings':<10} {'TR ID':<22} Organisation")
print("─" * 80)
for rank, (name, count) in enumerate(sorted_orgs[:10], 1):
    tr = org_tr_ids.get(name, "—")
    print(f"{rank:<5} {count:<10} {tr:<22} {name}")

top_orgs = [name for name, _ in sorted_orgs[:TOP_ORGS]]
print(f"\nSelected for analysis: {top_orgs}")

Rank  Meetings   TR ID                  Organisation
────────────────────────────────────────────────────────────────────────────────
1     16         90142503473-81         Vodafone Belgium SA
2     6          08957111909-85         Connect Europe
3     4          287872416724-91        Elisa Oyj
4     4          6822083232-32          Association of European Radios
5     4          35167875358-33         Nokia
6     4          02021363105-42         Ericsson
7     4          91216972728-77         BOUYGUES EUROPE
8     4          60052162589-72         Deutsche Telekom
9     4          041592911733-05        Centre on Regulation in Europe
10    2          366117914426-10        Amazon Europe Core SARL

Selected for analysis: ['Vodafone Belgium SA', 'Connect Europe', 'Elisa Oyj', 'Association of European Radios', 'Nokia', 'Ericsson', 'BOUYGUES EUROPE', 'Deutsche Telekom', 'Centre on Regulation in Europe', 'Amazon Europe Core SARL']


## Reciprocal Matching

In [138]:
def reciprocal_match(
    source_emb,
    source_texts,
    source_meta,
    target_emb,
    target_texts,
    top_k=TOP_K,
    recip_top_n=RECIP_TOP_N,
    forward_k=3,   # article must rank source in its top-K
    reverse_k=2,   # source must rank article in its top-K (was hardcoded top-1)
):
    """
    Find reciprocal semantic matches between source texts and target articles.

    Reciprocal: article A ranks source text S in its top `forward_k`,
    AND source text S ranks article A in its top `reverse_k`.

    Relaxing reverse_k > 1 catches cases where a source text is nearly tied
    between two articles — useful when there are many source texts (e.g. HYS chunks).

    Returns up to `recip_top_n` matches sorted by cosine score.
    """
    if source_emb is None or len(source_emb) == 0:
        return []

    # (n_source, n_articles) — cosine via dot product (embeddings are normalized)
    sim = source_emb @ target_emb.T

    # Precompute top-reverse_k articles for every source text
    top_arts_per_src = np.argsort(sim, axis=1)[:, ::-1][:, :reverse_k]  # (n_src, reverse_k)

    results, seen = [], set()

    for art_idx in range(len(target_texts)):
        # Top-forward_k source texts for this article
        a2s      = sim[:, art_idx]
        top_srcs = np.argsort(a2s)[::-1][:forward_k]

        for si in top_srcs:
            # Check reverse: is this article in the source's top-reverse_k?
            if art_idx not in top_arts_per_src[si]:
                continue

            key = (art_idx, int(si))
            if key in seen:
                continue
            seen.add(key)

            results.append({
                "article_idx":  art_idx,
                "source_idx":   int(si),
                "score":        float(a2s[si]),
                "source_text":  source_texts[si],
                "source_meta":  source_meta[si],
                "article":      proposal_articles[art_idx],
                "article_text": target_texts[art_idx],
            })

    results.sort(key=lambda x: x["score"], reverse=True)
    return results[:recip_top_n]


print("reciprocal_match() ready.  (forward_k=3, reverse_k=2 by default)")

reciprocal_match() ready.  (forward_k=3, reverse_k=2 by default)


In [139]:
org_results = {}

for org_name in top_orgs:
    tr_id = org_tr_ids.get(org_name)
    print(f"\n{'='*65}")
    print(f"Org: {org_name}")
    print(f"TR:  {tr_id or '—'}")

    # ── Commission meeting indices for this org ───────────────────────────────
    cm_idxs = [
        i for i, m in enumerate(cm_rows_v)
        if any(o["name"] == org_name for o in cm_meeting_to_orgs.get(m["id"], []))
    ]
    # ── Lobbying meeting indices for this org ─────────────────────────────────
    lm_idxs = [
        i for i, m in enumerate(lm_rows_v)
        if lm_orgs_map.get(m.get("organization_id"), {}).get("name") == org_name
    ]
    # ── Feedback indices (TR ID preferred, org name fallback) ─────────────────
    fb_idxs = [
        i for i, r in enumerate(fb_rows)
        if (tr_id and r.get("transparency_reg_id") == tr_id)
        or (not tr_id and (r.get("organisation_name") or "").lower() == org_name.lower())
    ]

    print(f"  Commission meeting points: {len(cm_idxs)}")
    print(f"  Lobbying meeting points:   {len(lm_idxs)}")
    print(f"  Feedback chunks:           {len(fb_idxs)}")

    # ── Build combined meeting embedding ──────────────────────────────────────
    emb_parts, text_parts, meta_parts = [], [], []

    if cm_idxs and cm_emb is not None:
        emb_parts.append(cm_emb[cm_idxs])
        text_parts += [cm_texts_v[i] for i in cm_idxs]
        meta_parts += [
            {
                "type":          "commission_meeting",
                "date":          cm_rows_v[i].get("meeting_date"),
                "commissioner":  cm_rows_v[i].get("commissioner_name"),
            }
            for i in cm_idxs
        ]
    if lm_idxs and lm_emb is not None:
        emb_parts.append(lm_emb[lm_idxs])
        text_parts += [lm_texts_v[i] for i in lm_idxs]
        meta_parts += [
            {"type": "lobbying_meeting", "date": lm_rows_v[i].get("meeting_date")}
            for i in lm_idxs
        ]

    all_mtg_emb   = np.vstack(emb_parts) if emb_parts else None
    all_mtg_texts = text_parts
    all_mtg_meta  = meta_parts

    # ── Feedback embedding for this org ───────────────────────────────────────
    org_fb_emb   = fb_emb[fb_idxs] if fb_idxs and fb_emb is not None else None
    org_fb_texts = [fb_rows[i]["chunk_text"] for i in fb_idxs]
    org_fb_meta  = [
        {"type": "feedback", "date": str(fb_rows[i].get("date_feedback") or "")[:10]}
        for i in fb_idxs
    ]

    # ── Run reciprocal matching ───────────────────────────────────────────────
    # Meetings: few texts, tight forward_k is fine
    meeting_matches = reciprocal_match(
        all_mtg_emb, all_mtg_texts, all_mtg_meta, article_emb, article_texts,
        forward_k=5, reverse_k=2,
    )

    # Feedback: many chunks — scale forward_k so relevant chunks aren't squeezed out
    fb_forward_k = max(10, len(fb_idxs) // 5) if fb_idxs else 10
    feedback_matches = reciprocal_match(
        org_fb_emb, org_fb_texts, org_fb_meta, article_emb, article_texts,
        forward_k=fb_forward_k, reverse_k=3,
    )

    print(f"  Reciprocal matches — meetings: {len(meeting_matches)},  feedback: {len(feedback_matches)}  (fb forward_k={fb_forward_k})")

    org_results[org_name] = {
        "tr_id":            tr_id,
        "meeting_count":    org_meeting_counts[org_name],
        "cm_count":         len(cm_idxs),
        "lm_count":         len(lm_idxs),
        "fb_count":         len(fb_idxs),
        "meeting_matches":  meeting_matches,
        "feedback_matches": feedback_matches,
    }


Org: Vodafone Belgium SA
TR:  90142503473-81
  Commission meeting points: 16
  Lobbying meeting points:   0
  Feedback chunks:           65
  Reciprocal matches — meetings: 3,  feedback: 3  (fb forward_k=13)

Org: Connect Europe
TR:  08957111909-85
  Commission meeting points: 6
  Lobbying meeting points:   0
  Feedback chunks:           0
  Reciprocal matches — meetings: 3,  feedback: 0  (fb forward_k=10)

Org: Elisa Oyj
TR:  287872416724-91
  Commission meeting points: 4
  Lobbying meeting points:   0
  Feedback chunks:           0
  Reciprocal matches — meetings: 3,  feedback: 0  (fb forward_k=10)

Org: Association of European Radios
TR:  6822083232-32
  Commission meeting points: 4
  Lobbying meeting points:   0
  Feedback chunks:           7
  Reciprocal matches — meetings: 3,  feedback: 3  (fb forward_k=10)

Org: Nokia
TR:  35167875358-33
  Commission meeting points: 4
  Lobbying meeting points:   0
  Feedback chunks:           24
  Reciprocal matches — meetings: 3,  feedback: 3

## LLM Interpretation

In [ ]:
def build_prompt_narrative(org_name: str, data: dict) -> str | None:
    all_matches = data["meeting_matches"] + data["feedback_matches"]
    if not all_matches:
        return None

    lines = [
        f"You are assessing whether the lobbying organisation **{org_name}** influenced",
        "the text of a European Commission legislative proposal.",
        "",
        "For each numbered match below you are given:",
        "  • ORG POSITION — what the organisation expressed in pre-proposal meetings or feedback",
        "  • ARTICLE      — the corresponding article or recital in the later Commission proposal",
        "",
        "For each match, classify as one of:",
        "  [ALIGNED]  — the org's position is reflected in the article",
        "  [OPPOSING] — the org's position conflicts with the article",
        "  [NOISE]    — topical overlap only; no meaningful position alignment",
        "",
        "End with a one-paragraph summary of overall alignment.",
        "If there is nothing meaningful to assess, reply only with: SKIP",
        "",
        "---",
    ]

    for i, m in enumerate(all_matches, 1):
        meta = m["source_meta"]
        art  = m["article"]

        src_type = meta.get("type", "unknown").replace("_", " ").title()
        src_date = str(meta.get("date") or "")[:10]
        commissioner = meta.get("commissioner", "")

        art_label = f"{art.get('element_type', '').title()} {art.get('element_number', '')}"
        if art.get("title"):
            art_label += f" — {art['title']}"

        src_label = src_type + (f", {src_date}" if src_date else "") + (f", {commissioner}" if commissioner else "")

        lines += [
            "",
            f"[{i}]  (cosine score: {m['score']:.3f})",
            f"  ORG POSITION ({src_label}):",
            m["source_text"][:600],
            "",
            f"  ARTICLE ({art_label}):",
            m["article_text"][:600],
        ]

    return "\n".join(lines)


print("build_prompt_narrative() ready.")

In [148]:
llm_responses    = {}
no_position_orgs = []
no_match_orgs    = []

for org_name in top_orgs:
    data   = org_results[org_name]
    prompt = build_prompt_narrative(org_name, data)

    if prompt is None:
        no_position_orgs.append((org_name, data["meeting_count"]))
        continue

    print(f"Calling {LLM_MODEL} for: {org_name} ...")
    msg = anth.messages.create(
        model=LLM_MODEL,
        max_tokens=800,
        messages=[{"role": "user", "content": prompt}],
    )
    response = msg.content[0].text.strip()
    if response == "SKIP":
        no_match_orgs.append((org_name, data["meeting_count"]))
    else:
        llm_responses[org_name] = (data, response)   # always store as tuple

print(f"\nOrgs with matches : {len(llm_responses)}")


Calling claude-sonnet-4-6 for: Vodafone Belgium SA ...
Calling claude-sonnet-4-6 for: Connect Europe ...
Calling claude-sonnet-4-6 for: Elisa Oyj ...
Calling claude-sonnet-4-6 for: Association of European Radios ...
Calling claude-sonnet-4-6 for: Nokia ...
Calling claude-sonnet-4-6 for: Ericsson ...
Calling claude-sonnet-4-6 for: BOUYGUES EUROPE ...
Calling claude-sonnet-4-6 for: Deutsche Telekom ...
Calling claude-sonnet-4-6 for: Centre on Regulation in Europe ...
Calling claude-sonnet-4-6 for: Amazon Europe Core SARL ...

Orgs with matches : 8


## Final Report

In [ ]:
SEP  = "=" * 80
THIN = "─" * 80

def extract_matches(text):
    """Pull lines that contain [ALIGNED] or [OPPOSING] anywhere."""
    return [
        line.strip() for line in text.split("\n")
        if "[ALIGNED]" in line or "[OPPOSING]" in line
    ]

for org_name, (data, text) in llm_responses.items():
    matches = extract_matches(text)
    if not matches:
        continue

    submitted_feedback = "submitted feedback" if data["fb_count"] > 0 else "did not submit feedback"
    meeting_count = data["meeting_count"]

    print(SEP)
    print(
        f"{org_name} — {meeting_count} pre-proposal meeting{'s' if meeting_count > 1 else ''} "
        f"with the Commission, {submitted_feedback}.\n"
    )
    print(
        "Within the disclosed feedback and meeting summaries, the following pre-proposal "
        "positions of the organisation match articles in the later drafted legislative proposal. "
        "This suggests influence, but does not prove it. It only proves alignment between the "
        "organisation's expressed wishes and the specific article in the regulation.\n"
    )
    for line in matches:
        print(line)
        print()

print(SEP)

In [147]:
for k, v in llm_responses.items():
    print(k, type(v), repr(v)[:80])


Vodafone Belgium SA <class 'str'> '## Classification of Matches\n\n**[1] & [2] — NOISE**\nThe org text raises broa
Connect Europe <class 'str'> "Here are the classified matches:\n\n---\n\n**[1] ALIGNED**\nConnect Europe memb
Elisa Oyj <class 'str'> '## Classification of Matches\n\n**[1], [2], [3]** — All three matches are dupli
Association of European Radios <class 'str'> "## Classification of Matches\n\n**[1] & [2] — NOISE**\nThe org text is a meetin
Nokia <class 'str'> "## Classification of Matches\n\n**[1], [2], [3] — ALIGNED**\nNokia's emphasis o
Ericsson <class 'str'> "I'll analyse each match for topic alignment and position compatibility.\n\n---\
BOUYGUES EUROPE <class 'str'> "## Classification of Matches\n\n**[1] ALIGNED**\nBouygues stressed that effecti
Deutsche Telekom <class 'str'> "I'll analyse each match and classify it, then provide a summary.\n\n---\n\n## M
Centre on Regulation in Europe <class 'str'> "## Classification of Matches\n\n**[1], [2], [3] — ALIGNED**\nAll three

## LLM Interpretation — Structured (tool use, per-org, → DataFrame)

In [57]:
# ── Tool schema + prompt builder ──────────────────────────────────────────────

CLASSIFY_TOOL = {
    "name": "classify_matches",
    "description": (
        "Classify each numbered match between an organisation's pre-proposal text "
        "and a proposal article. Use ALIGNED if the org's expressed position is "
        "reflected in the article, OPPOSING if it conflicts, NOISE if the overlap "
        "is coincidental or purely topical with no position alignment."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "classifications": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "match_idx":  {"type": "integer", "description": "1-based index from the prompt"},
                        "label":      {"type": "string",  "enum": ["ALIGNED", "OPPOSING", "NOISE"]},
                        "reasoning":  {"type": "string",  "description": "One or two sentences explaining the classification."},
                    },
                    "required": ["match_idx", "label", "reasoning"],
                },
            }
        },
        "required": ["classifications"],
    },
}


def build_prompt_structured(org_name: str, data: dict) -> str | None:
    """Build a prompt that lists all matches for one org and asks for per-match JSON."""
    all_matches = [
        (m, "meeting")  for m in data["meeting_matches"]
    ] + [
        (m, "feedback") for m in data["feedback_matches"]
    ]
    if not all_matches:
        return None

    lines = [
        f"Organisation: {org_name}",
        f"Pre-proposal meetings with Commission: {data['meeting_count']}",
        "",
        "Below are semantic matches between this organisation's pre-proposal texts "
        "and articles in the later Commission proposal. "
        "Classify each match and return exactly "
        f"{len(all_matches)} classification object(s) via the tool.",
        "",
    ]

    for idx, (m, source_kind) in enumerate(all_matches, 1):
        meta = m["source_meta"]
        art  = m["article"]

        source_label = meta.get("type", source_kind).replace("_", " ").title()
        date_str     = str(meta.get("date") or "")[:10]
        commissioner = meta.get("commissioner", "")

        art_label = f"{art.get('element_type','').title()} {art.get('element_number','')}"
        if art.get("title"):
            art_label += f" — {art['title']}"

        lines += [
            f"[{idx}] cosine score: {m['score']:.3f}",
            f"Source ({source_label}{', ' + date_str if date_str else ''}"
            + (f", {commissioner}" if commissioner else "") + "):",
            m["source_text"][:600],
            "",
            f"Article ({art_label}):",
            m["article_text"][:600],
            "",
        ]

    return "\n".join(lines)


print("Tool schema and prompt builder ready.")
print(f"  Tool: {CLASSIFY_TOOL['name']}")

Tool schema and prompt builder ready.
  Tool: classify_matches


In [58]:
# ── LLM loop — one tool-use call per org ──────────────────────────────────────
import pandas as pd

raw_rows = []   # one dict per match, filled in below

for org_name in top_orgs:
    data    = org_results[org_name]
    prompt  = build_prompt_structured(org_name, data)

    if prompt is None:
        print(f"  {org_name}: no matches — skipped")
        continue

    all_matches = data["meeting_matches"] + data["feedback_matches"]
    n           = len(all_matches)

    print(f"Calling {LLM_MODEL} for: {org_name}  ({n} match{'es' if n != 1 else ''}) ...", end=" ", flush=True)

    msg = anth.messages.create(
        model=LLM_MODEL,
        max_tokens=1200,
        tools=[CLASSIFY_TOOL],
        tool_choice={"type": "any"},   # force tool use
        messages=[{"role": "user", "content": prompt}],
    )

    # Extract tool_use block
    tool_block = next((b for b in msg.content if b.type == "tool_use"), None)
    if tool_block is None:
        print("WARNING: no tool_use block returned")
        continue

    classifications = tool_block.input.get("classifications", [])
    print(f"got {len(classifications)} classification(s)")

    # Build one row per match
    source_kind_list = (
        ["meeting"] * len(data["meeting_matches"]) +
        ["feedback"] * len(data["feedback_matches"])
    )

    cls_by_idx = {c["match_idx"]: c for c in classifications}

    for i, (m, source_kind) in enumerate(zip(all_matches, source_kind_list), 1):
        meta = m["source_meta"]
        art  = m["article"]
        cls  = cls_by_idx.get(i, {})

        raw_rows.append({
            "organisation":       org_name,
            "tr_id":              data["tr_id"],
            "meeting_count":      data["meeting_count"],
            "source_kind":        source_kind,
            "source_type":        meta.get("type", source_kind),
            "source_date":        str(meta.get("date") or "")[:10] or None,
            "source_commissioner":meta.get("commissioner"),
            "source_text":        m["source_text"],
            "article_type":       art.get("element_type"),
            "article_number":     art.get("element_number"),
            "article_title":      art.get("title"),
            "article_text":       m["article_text"],
            "cosine_score":       round(m["score"], 4),
            "label":              cls.get("label"),
            "reasoning":          cls.get("reasoning"),
        })

print(f"\nDone. {len(raw_rows)} total rows collected.")

NameError: name 'top_orgs' is not defined

In [ ]:
# ── Build DataFrame + report ───────────────────────────────────────────────────
matches_df = pd.DataFrame(raw_rows)

# Label distribution
print("Label distribution:")
print(matches_df["label"].value_counts().to_string())
print()

# Summary per org (non-noise only)
signal = matches_df[matches_df["label"] != "NOISE"].copy()

print(f"{'='*80}")
print(f"Non-noise matches: {len(signal)} / {len(matches_df)} total")
print(f"{'='*80}\n")

for org_name, grp in signal.groupby("organisation", sort=False):
    mc = grp["meeting_count"].iloc[0]
    print(f"{org_name}  ({mc} pre-proposal meeting{'s' if mc != 1 else ''})")
    for _, row in grp.iterrows():
        art = f"{row['article_type']} {row['article_number']}"
        if row["article_title"]:
            art += f" — {row['article_title']}"
        print(f"  [{row['label']}]  {art}  (score={row['cosine_score']})")
        print(f"    {row['reasoning']}")
    print()

# Full table (optional — shows all columns)
matches_df

In [ ]:
# ── Export to JSON ─────────────────────────────────────────────────────────────
import json, pathlib

proc_slug = PROCEDURE_ID.replace("/", "_").replace("(", "").replace(")", "")
out_path  = pathlib.Path("results") / f"preproposal_matches_{proc_slug}.json"
out_path.parent.mkdir(exist_ok=True)

matches_df.to_json(out_path, orient="records", indent=2, force_ascii=False)
print(f"Saved {len(matches_df)} rows → {out_path}")